In [56]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

In [57]:
df = pd.read_csv('https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv')
df.head()

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


In [58]:
df = df[['engine_displacement','horsepower','vehicle_weight','model_year','fuel_efficiency_mpg']]
df.head()

,engine_displacement,horsepower,vehicle_weight,model_year,fuel_efficiency_mpg
0,2180,243.0,3870,2006,31.9
1,2390,272.0,4210,2008,31.3
2,2320,267.0,4240,1996,27.5
3,2130,258.0,4490,1989,28.5
4,2580,304.0,4510,1994,31.0


In [59]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 5 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   engine_displacement  10000 non-null  int64  
 1   horsepower           9123 non-null   float64
 2   vehicle_weight       10000 non-null  int64  
 3   model_year           10000 non-null  int64  
 4   fuel_efficiency_mpg  10000 non-null  float64
dtypes: float64(2), int64(3)
memory usage: 390.8 KB


In [60]:
df.horsepower.median()

254.0

In [61]:
df['horsepower'] = df['horsepower'].fillna(0)

In [62]:
n= len(df)

n_val = int(n*0.2)
n_test = int(n*0.2)
n_train = (n - n_val - n_test)
n_train, n_val, n_test

(6000, 2000, 2000)

In [63]:
np.random.seed(42)
idx = np.arange(n)
np.random.shuffle(idx)

In [64]:
df_val = df.iloc[:n_val]
df_test = df.iloc[n_val:n_val+n_test]
df_train = df.iloc[n_val+n_test:]

In [65]:
df_train.head()

,engine_displacement,horsepower,vehicle_weight,model_year,fuel_efficiency_mpg
4000,2450,262.0,4530,1980,24.8
4001,2310,258.0,4420,2017,32.7
4002,2010,217.0,3880,2022,36.7
4003,2330,0.0,4410,1980,29.8
4004,2490,262.0,4420,2023,29.1


In [66]:
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

### New_Fresh

In [67]:
def train_linear_regression(X,y):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])
    
    XTX = X.T.dot(X)
    XTX_inv = np.linalg.inv(XTX)
    w_full = XTX_inv.dot(X.T).dot(y)
    
    return w_full[0], w_full[1:]

In [68]:
base_columns = ['engine_displacement','horsepower',	'vehicle_weight','model_year']

In [69]:
y_train = df_train['fuel_efficiency_mpg'] 
y_val = df_val['fuel_efficiency_mpg'] 

In [70]:
def rmse(y, y_pred):
    se = (y - y_pred)**2
    mse = se.mean()
    return np.sqrt(mse)
    

In [71]:
def prepare_X_0(df):
    df_num = df[base_columns]
    df_num = df_num.fillna(0)
    X = df_num.values
    return X

In [72]:
X_train = prepare_X_0(df_train)
w0, w = train_linear_regression(X_train, y_train)

X_val = prepare_X_0(df_val)
y_pred = w0 + X_val.dot(w)

rmse(y_val, y_pred)

np.float64(2.1796402127885584)

In [73]:
def prepare_X_mean(df):
    df_num = df[base_columns]
    df_num['horsepower'] = df_num['horsepower'].fillna(df_num['horsepower'].mean())
    X = df_num.values
    return X

In [74]:
X_train = prepare_X_0(df_train)
w0, w = train_linear_regression(X_train, y_train)

X_val = prepare_X_mean(df_val)
y_pred = w0 + X_val.dot(w)

rmse(y_val, y_pred)

C:\Users\omoni\AppData\Local\Temp\ipykernel_25240\3654317275.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_num['horsepower'] = df_num['horsepower'].fillna(df_num['horsepower'].mean())


np.float64(2.1796402127885584)

In [76]:
def train_linear_regression(X,y, r=0.001):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])
    
    XTX = X.T.dot(X)
    XTX = XTX + r*np.eye(XTX.shape[0])
    
    XTX_inv = np.linalg.inv(XTX)
    w_full = XTX_inv.dot(X.T).dot(y)
    
    return w_full[0], w_full[1:]

In [83]:
for r in [0,0.01,0.1,1,5,10,100]:
    X_train = prepare_X_0(df_train)
    w0, w = train_linear_regression(X_train, y_train , r)

    X_val = prepare_X_0(df_val)
    y_pred = w0 + X_val.dot(w)

    score = round(rmse(y_val, y_pred),4)
    
    print(r, w0, score)

0 -156.71745139978918 2.1796
0.01 -151.0051230054973 2.1795
0.1 -113.70450671446831 2.1952
1 -32.76639218369987 2.3237
5 -7.869561379408848 2.3869
10 -4.036111394009929 2.3975
100 -0.4131768876480977 2.4077


In [ ]:
n=len(df)

n_val = int(n*0.2)
n_test = int(n*0.2)
n_train = (n - n_val - n_test)
n_train, n_val, n_test

In [92]:
con = [ ]

for s in [0,1,2,3,4,5,6,7,8,9]:
    np.random.seed(s)
    idx = np.arange(n)
    np.random.shuffle(idx)
    
    df_val = df.iloc[:n_val]
    df_test = df.iloc[n_val:n_val+n_test]
    df_train = df.iloc[n_val+n_test:]

    X_train = prepare_X_0(df_train)
    w0, w = train_linear_regression(X_train, y_train , 1)

    X_val = prepare_X_0(df_val)
    y_pred = w0 + X_val.dot(w)

    score = round(rmse(y_val, y_pred),4)
    con.append(score)
    
    print(score)
    

2.3237
2.3237
2.3237
2.3237
2.3237
2.3237
2.3237
2.3237
2.3237
2.3237


In [89]:
np.random.seed(9)
idx = np.arange(n)
np.random.shuffle(idx)

df_val = df.iloc[:n_val]
df_test = df.iloc[n_val:n_val+n_test]
df_train = df.iloc[n_val+n_test:]

X_train = prepare_X_0(df_train)
w0, w = train_linear_regression(X_train, y_train , 0.001)

X_val = prepare_X_0(df_val)
y_pred = w0 + X_val.dot(w)

score = round(rmse(y_val, y_pred),4)
score

np.float64(2.1796)